# 🌌 Earth Field Analysis
## Layer 0 – External Cosmic Drivers

| Layer | Name | Function |
|-------|------|----------|
| **0** | **External Cosmic Drivers** | **Outer modulator – this layer** |
| 1 | Planetary Body | Geophysical base |
| 2 | Surface / Oceans / Land | Surface zone |
| 3 | Atmosphere / Weather / Thunderstorms | Weather dynamics |
| 4 | Ionosphere | Electrical layer |
| 5 | Global Electric Circuit | Field coupling |
| 6 | Resonance Field / Schumann | Patterns |
| 7 | Earth Field State Engine | Synthesis |
| 8 | Research / Hypotheses | Analysis & patterns |

> **Core question:** Which external influences change the electromagnetic and atmospheric states of the Earth?


In [1]:
import warnings; warnings.filterwarnings('ignore')
import datetime, json, math, requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import re

print(f'Packages loaded')
print(f'Analysis date: {datetime.date.today()}')

Packages loaded
Analysis date: 2026-07-23


---
## 1. System Structure: The 6 Elements of Layer 0

In [2]:
elements = [
    {'name': 'Solar Radiation<br>(TSI)',     'x': 0.50, 'y': 0.90, 'color': '#F2A623', 'sz': 60},
    {'name': 'UV / X-Ray',                   'x': 0.18, 'y': 0.74, 'color': '#E85D24', 'sz': 54},
    {'name': 'Solar Wind<br>(IMF)',           'x': 0.82, 'y': 0.74, 'color': '#F2A623', 'sz': 54},
    {'name': 'Geomag. Storms<br>(CME, Kp)',  'x': 0.10, 'y': 0.51, 'color': '#7F77DD', 'sz': 56},
    {'name': 'Cosmic<br>Radiation (GCR)',    'x': 0.90, 'y': 0.51, 'color': '#378ADD', 'sz': 56},
    {'name': 'Planetary Cycles<br>(Milankovitch)', 'x': 0.50, 'y': 0.54, 'color': '#639922', 'sz': 54},
]
earth = {'x': 0.50, 'y': 0.16}

fig = go.Figure()
for el in elements:
    dx = earth['x'] - el['x']; dy = earth['y'] - el['y']
    dist = math.sqrt(dx**2 + dy**2)
    t = 0.06 / dist
    ex, ey = earth['x'] - dx * t, earth['y'] - dy * t
    fig.add_trace(go.Scatter(
        x=[el['x'], ex], y=[el['y'], ey], mode='lines',
        line=dict(color=el['color'], width=2), opacity=0.5,
        showlegend=False, hoverinfo='skip'
    ))

fig.add_trace(go.Scatter(
    x=[earth['x']], y=[earth['y']], mode='markers+text',
    marker=dict(size=82, color='#1D9E75', opacity=0.90, line=dict(color='white', width=2.5)),
    text=['🌍 Earth System<br>Layer 1–7'], textposition='middle center',
    textfont=dict(size=10, color='white'), showlegend=False, hoverinfo='skip'
))

for el in elements:
    fig.add_trace(go.Scatter(
        x=[el['x']], y=[el['y']], mode='markers+text',
        marker=dict(size=el['sz'], color=el['color'], opacity=0.90,
                    line=dict(color='white', width=2)),
        text=[el['name']], textposition='middle center',
        textfont=dict(size=9.5, color='white'), showlegend=False,
        hovertemplate=el['name'].replace('<br>',' ') + '<extra></extra>'
    ))

for txt, px, py, col in [
    ('☀️  Electromagnetic Radiation', 0.50, 0.99, '#B05010'),
    ('⚡  Particles & Fields',         0.50, 0.61, '#534AB7'),
    ('🔄  Orbital / Long-term',        0.50, 0.44, '#3B6D11'),
]:
    fig.add_annotation(x=px, y=py, text=txt, showarrow=False,
                       xref='paper', yref='paper',
                       font=dict(size=11, color=col))

fig.update_layout(
    title=dict(text='Layer 0 – External Drivers: System Structure', font=dict(size=16)),
    xaxis=dict(showgrid=False, zeroline=False, visible=False, range=[-0.05, 1.05]),
    yaxis=dict(showgrid=False, zeroline=False, visible=False, range=[0.03, 1.05]),
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    height=530, margin=dict(l=20, r=20, t=55, b=20)
)
fig.show()

---
## 2. Real-Time Data – NOAA SWPC

In [3]:
# ============================================================
# REAL-TIME DATA – NOAA SWPC Public APIs (no API key required)
# ============================================================

ENDPOINTS = {
    'kp_1m':      'https://services.swpc.noaa.gov/json/planetary_k_index_1m.json',
    'f107_cycle': 'https://services.swpc.noaa.gov/json/solar-cycle/observed-solar-cycle-indices.json',
    'solar_wind': 'https://services.swpc.noaa.gov/json/rtsw/rtsw_wind_1m.json',
    'mag_field':  'https://services.swpc.noaa.gov/json/rtsw/rtsw_mag_1m.json',
    'xray':       'https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json',
    'alerts':     'https://services.swpc.noaa.gov/products/alerts.json',
}

def fetch(key, timeout=15):
    url = ENDPOINTS[key]
    try:
        r = requests.get(url, timeout=timeout); r.raise_for_status()
        data = r.json()
        print(f'  OK {key:<14} {len(data):>5} entries')
        return data
    except Exception as e:
        print(f'  ERR {key:<13} {e}')
        return None

print('Loading NOAA SWPC data...')
raw = {k: fetch(k) for k in ENDPOINTS}
print(f'\n📥 {sum(1 for v in raw.values() if v)}/{len(ENDPOINTS)} sources loaded')

Loading NOAA SWPC data...
  OK kp_1m            356 entries


  OK f107_cycle      3330 entries
  OK solar_wind      3729 entries
  OK mag_field       3689 entries
  OK xray            2876 entries
  OK alerts           130 entries

📥 6/6 sources loaded


In [4]:
# ============================================================
# DATA PROCESSING – robust column detection
# ============================================================

import re

def find_time_col(df):
    return next((c for c in df.columns if 'time' in c.lower()), df.columns[0])

# --- Kp ---
# API delivers: time_tag, kp_index, estimated_kp, kp
df_kp = None
if raw['kp_1m']:
    df_kp = pd.DataFrame(raw['kp_1m'])
    print('Kp columns:', df_kp.columns.tolist())
    tc = find_time_col(df_kp)
    kp_candidates = [c for c in df_kp.columns if 'kp' in c.lower() and c != tc]
    kc = 'kp' if 'kp' in df_kp.columns else kp_candidates[-1]
    print(f'  Using: time={tc!r}, kp={kc!r}')
    print(f'  Sample values [{kc}]:', df_kp[kc].head(3).tolist())
    df_kp = df_kp[[tc, kc]].copy()
    df_kp.columns = ['time', 'kp']
    df_kp['time'] = pd.to_datetime(df_kp['time'])
    # NOAA encodes Kp as '2M', '3+', '1-' – extract numeric part
    df_kp['kp'] = df_kp['kp'].astype(str).str.extract(r'([0-9]+(?:\.[0-9]*)?)')[0]
    df_kp['kp'] = pd.to_numeric(df_kp['kp'], errors='coerce')
    df_kp = df_kp[df_kp['kp'] >= 0].dropna(subset=['kp']).sort_values('time').tail(1440)
    if df_kp.empty:
        print('  ⚠️  Kp DataFrame empty after cleaning – trying kp_index column')
        df_kp = pd.DataFrame(raw['kp_1m'])[[tc, 'kp_index']].copy()
        df_kp.columns = ['time', 'kp']
        df_kp['time'] = pd.to_datetime(df_kp['time'])
        df_kp['kp']   = pd.to_numeric(df_kp['kp'], errors='coerce')
        df_kp = df_kp[df_kp['kp'] >= 0].dropna(subset=['kp']).sort_values('time').tail(1440)
    if not df_kp.empty:
        print(f'Kp: current {df_kp["kp"].iloc[-1]:.2f}, max (24h) {df_kp["kp"].max():.2f}')
    else:
        print('  ERR Kp data unusable'); df_kp = None

# --- F10.7 ---
df_f107 = None
if raw['f107_cycle']:
    df_f107 = pd.DataFrame(raw['f107_cycle'])
    print('F10.7 columns:', df_f107.columns.tolist())
    tc = find_time_col(df_f107)
    df_f107['time'] = pd.to_datetime(df_f107[tc])
    f107_col = next((c for c in df_f107.columns
                     if ('f10' in c.lower() or 'flux' in c.lower())
                     and 'smooth' not in c.lower()), None)
    if f107_col is None:
        print('  ⚠️  No F10.7 column found')
    else:
        df_f107 = df_f107.rename(columns={f107_col: 'f10.7'})
        for col in ['f10.7', 'ssn', 'smoothed_ssn']:
            if col in df_f107.columns:
                df_f107[col] = pd.to_numeric(df_f107[col], errors='coerce')
        sm_col = next((c for c in df_f107.columns
                       if 'smooth' in c.lower() and 'f10' in c.lower()), None)
        if sm_col:
            df_f107 = df_f107.rename(columns={sm_col: 'smoothed_f10.7'})
            df_f107['smoothed_f10.7'] = pd.to_numeric(df_f107['smoothed_f10.7'], errors='coerce')
        df_f107 = df_f107.dropna(subset=['f10.7']).sort_values('time').tail(60)
        print(f'F10.7: current {df_f107["f10.7"].iloc[-1]:.1f} sfu')

# --- Solar Wind ---
df_sw = None
if raw['solar_wind']:
    df_sw = pd.DataFrame(raw['solar_wind'])
    tc = find_time_col(df_sw)
    df_sw['time'] = pd.to_datetime(df_sw[tc])
    for col in ['speed', 'density', 'temperature']:
        if col in df_sw.columns:
            df_sw[col] = pd.to_numeric(df_sw[col], errors='coerce')
    df_sw = df_sw.sort_values('time').tail(1440)
    if 'speed' in df_sw.columns:
        print(f'SW: current {df_sw["speed"].dropna().iloc[-1]:.0f} km/s')

# --- IMF / Magnetic Field ---
df_mag = None
if raw['mag_field']:
    df_mag = pd.DataFrame(raw['mag_field'])
    tc = find_time_col(df_mag)
    df_mag['time'] = pd.to_datetime(df_mag[tc])
    for col in ['bt', 'bz_gsm', 'bx_gsm', 'by_gsm']:
        if col in df_mag.columns:
            df_mag[col] = pd.to_numeric(df_mag[col], errors='coerce')
    df_mag = df_mag.sort_values('time').tail(1440)
    if 'bz_gsm' in df_mag.columns:
        bz = df_mag['bz_gsm'].dropna().iloc[-1]
        direction = 'southward ⚠️' if bz < -5 else 'northward ✅' if bz > 0 else 'neutral'
        print(f'Bz: current {bz:.1f} nT  ({direction})')

# --- X-Ray ---
df_xray = None
if raw['xray']:
    df_xray = pd.DataFrame(raw['xray'])
    tc = find_time_col(df_xray)
    df_xray['time'] = pd.to_datetime(df_xray[tc])
    fc = next((c for c in df_xray.columns
               if any(k in c.lower() for k in ['flux', 'long', 'energy'])
               and c != tc), df_xray.columns[1])
    df_xray['flux'] = pd.to_numeric(df_xray[fc], errors='coerce')
    df_xray = df_xray.sort_values('time')
    print(f'X-Ray: {len(df_xray)} entries')

print('\nData processing complete')

Kp columns: ['time_tag', 'kp_index', 'estimated_kp', 'kp']
  Using: time='time_tag', kp='kp'
  Sample values [kp]: ['1M', '1M', '1Z']
Kp: current 0.00, max (24h) 3.00
F10.7 columns: ['time-tag', 'ssn', 'smoothed_ssn', 'observed_swpc_ssn', 'smoothed_swpc_ssn', 'f10.7', 'smoothed_f10.7']
F10.7: current 138.2 sfu
Bz: current -3.9 nT  (neutral)
X-Ray: 2876 entries

Data processing complete


---
## 3. Time-Series Dashboard (Real Data)

In [5]:
# ============================================================
# DASHBOARD – Kp / Solar Wind / IMF Bz / X-Ray
# ============================================================

panels = [
    (df_kp,   'kp',     'Kp-Index',                       '#7F77DD', 'bar'),
    (df_sw,   'speed',  'Solar Wind Speed [km/s]',         '#F2A623', 'line'),
    (df_mag,  'bz_gsm', 'IMF Bz GSM [nT]',                '#378ADD', 'bar'),
    (df_xray, 'flux',   'GOES X-Ray Flux [W/m²] (log)',    '#E85D24', 'line'),
]
active = [(df, col, title, color, kind)
          for df, col, title, color, kind in panels
          if df is not None and col in df.columns]

if not active:
    print('No data available.')
else:
    fig = make_subplots(
        rows=len(active), cols=1, shared_xaxes=True,
        subplot_titles=[p[2] for p in active],
        vertical_spacing=0.06
    )
    for i, (df, col, title, color, kind) in enumerate(active, 1):
        ds = df.dropna(subset=[col])
        if kind == 'bar':
            if col == 'kp':
                mc = ['#2ecc71' if v < 4 else '#f39c12' if v < 6 else '#e74c3c'
                      for v in ds[col]]
            elif col == 'bz_gsm':
                mc = ['#e74c3c' if v < -5 else '#2ecc71' if v > 5 else '#95a5a6'
                      for v in ds[col]]
            else:
                mc = color
            fig.add_trace(go.Bar(x=ds['time'], y=ds[col], marker_color=mc,
                                 name=title, opacity=0.75), row=i, col=1)
        else:
            h = color.lstrip('#')
            r_c, g_c, b_c = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
            fc = f'rgba({r_c},{g_c},{b_c},0.12)'
            fig.add_trace(go.Scatter(x=ds['time'], y=ds[col], name=title,
                                     line=dict(color=color, width=1.5),
                                     fill='tozeroy', fillcolor=fc), row=i, col=1)

        if col == 'kp':
            fig.add_hline(y=5, line_dash='dot', line_color='#e74c3c',
                          annotation_text='G1 Storm threshold', row=i, col=1)
            fig.update_yaxes(range=[0, 9], row=i, col=1)
        elif col == 'speed':
            fig.add_hline(y=500, line_dash='dot', line_color='#E85D24',
                          annotation_text='Elevated wind', row=i, col=1)
        elif col == 'bz_gsm':
            fig.add_hline(y=0, line_color='gray', line_width=0.5, row=i, col=1)
            fig.add_hline(y=-5, line_dash='dot', line_color='#e74c3c',
                          annotation_text='Coupling threshold', row=i, col=1)
        elif col == 'flux':
            fig.update_yaxes(type='log', row=i, col=1)
            for lvl, lbl, lc in [(1e-4,'X','#c0392b'),(1e-5,'M','#e67e22'),(1e-6,'C','#f1c40f')]:
                fig.add_hline(y=lvl, line_dash='dot', line_color=lc,
                              annotation_text=lbl, row=i, col=1)

    fig.update_layout(
        title=dict(text='Layer 0 – Real Data Dashboard (NOAA SWPC, last 24h)', font=dict(size=16)),
        height=180 * len(active) + 100,
        showlegend=False,
        plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=70, r=30, t=60, b=40)
    )
    fig.show()

---
## 4. Solar Cycle – F10.7 & Sunspot Number

In [6]:
if df_f107 is not None:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=['F10.7 Radio Flux [sfu]', 'Sunspot Number (SSN)'],
                        vertical_spacing=0.1)

    fig.add_trace(go.Scatter(x=df_f107['time'], y=df_f107['f10.7'],
                             line=dict(color='#F2A623', width=2), name='F10.7',
                             fill='tozeroy', fillcolor='rgba(242,166,35,0.15)'), row=1, col=1)
    if 'smoothed_f10.7' in df_f107.columns:
        fig.add_trace(go.Scatter(x=df_f107['time'], y=df_f107['smoothed_f10.7'],
                                 line=dict(color='#E85D24', width=2.5, dash='dash'),
                                 name='F10.7 smoothed'), row=1, col=1)

    if 'ssn' in df_f107.columns:
        fig.add_trace(go.Bar(x=df_f107['time'], y=df_f107['ssn'],
                             marker_color='#7F77DD', opacity=0.55, name='SSN'), row=2, col=1)
    if 'smoothed_ssn' in df_f107.columns:
        fig.add_trace(go.Scatter(x=df_f107['time'], y=df_f107['smoothed_ssn'],
                                 line=dict(color='#534AB7', width=2.5), name='SSN smoothed'), row=2, col=1)

    fig.add_hline(y=150, line_dash='dot', line_color='#E85D24',
                  annotation_text='High activity', row=1, col=1)
    fig.update_yaxes(title_text='F10.7 [sfu]', row=1, col=1)
    fig.update_yaxes(title_text='SSN', row=2, col=1)
    fig.update_layout(
        title=dict(text='Solar Cycle 25 – F10.7 & SSN (monthly means, last 60 months)', font=dict(size=15)),
        height=480, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=70, r=30, t=60, b=40)
    )
    fig.show()

---
## 5. Active NOAA Space Weather Alerts

In [7]:
if raw['alerts']:
    alerts = [a for a in raw['alerts'] if isinstance(a, dict)]
    print(f'NOAA Space Weather Alerts: {len(alerts)} messages\n')
    if alerts:
        for a in alerts[:6]:
            msg = a.get('message', str(a))
            print('─' * 60)
            print(msg[:400])
    else:
        print('No active alerts.')
else:
    print('Alerts unavailable.')

NOAA Space Weather Alerts: 130 messages

────────────────────────────────────────────────────────────
Space Weather Message Code: WATA20
Serial Number: 1117
Issue Time: 2026 Jul 22 1925 UTC

WATCH: Geomagnetic Storm Category G1 Predicted 
Highest Storm Level Predicted by Day:
Jul 23:  None (Below G1)   Jul 24:  G1 (Minor)   Jul 25:  None (Below G1)   
THIS SUPERSEDES ANY/ALL PRIOR WATCHES IN EFFECT
Comment: 

NOAA Space Weather Scale descriptions can be found at
www.swpc.noaa.gov/noaa-scale
────────────────────────────────────────────────────────────
Space Weather Message Code: SUM10R
Serial Number: 924
Issue Time: 2026 Jul 22 1750 UTC

SUMMARY: 10cm Radio Burst 
Begin Time: 2026 Jul 22 1552 UTC
Maximum Time: 2026 Jul 22 1552 UTC
End Time: 2026 Jul 22 1552 UTC
Peak Flux: 150 sfu
Duration: 1 minutes
Latest Penticton Noon Flux: 149 sfu
Comment: 
────────────────────────────────────────────────────────────
Space Weather Message Code: WARK04
Serial Number: 5390
Issue Time: 2026 Jul 22 1456

---
## 6. State Assessment & Handoff to Layer 1

In [8]:
# ============================================================
# STEP 1 – Collect raw values with source modes
# Mode: 'primary' | 'fallback_3h' | 'last_valid' | 'missing'
# ============================================================

def safe_last(df, col):
    if df is None or col not in df.columns: return None
    s = df[col].dropna()
    return float(s.iloc[-1]) if not s.empty else None

f107_now = safe_last(df_f107, 'f10.7')
bz_now   = safe_last(df_mag,  'bz_gsm')
kp_now   = safe_last(df_kp,   'kp')
sw_now   = safe_last(df_sw,   'speed')

source_modes = {
    'F10.7':    'primary'  if f107_now is not None else 'missing',
    'Kp':       'primary'  if kp_now   is not None else 'missing',
    'IMF_Bz':   'primary'  if bz_now   is not None else 'missing',
    'SW_speed': 'primary'  if sw_now   is not None else 'missing',
}

# Kp Fallback – 3h endpoint
if kp_now is None:
    try:
        r = requests.get(
            'https://services.swpc.noaa.gov/json/noaa-planetary-k-index.json',
            timeout=10)
        r.raise_for_status()
        fb = pd.DataFrame(r.json())
        tc = next((c for c in fb.columns if 'time' in c.lower()), fb.columns[0])
        kc = next((c for c in fb.columns if 'kp' in c.lower() and c != tc), None)
        if kc:
            fb[kc] = fb[kc].astype(str).str.extract(r'([0-9]+(?:\.[0-9]*)?)')[0]
            fb[kc] = pd.to_numeric(fb[kc], errors='coerce')
            fb = fb[fb[kc] >= 0].dropna(subset=[kc])
            if not fb.empty:
                kp_now = float(fb[kc].iloc[-1])
                source_modes['Kp'] = 'fallback_3h'
                print(f'  Kp fallback (3h): {kp_now:.1f}')
    except Exception as e:
        print(f'  Kp fallback failed: {e}')

# Solar Wind Fallback – last valid value
if sw_now is None and df_sw is not None and 'speed' in df_sw.columns:
    s = df_sw['speed'].dropna()
    if not s.empty:
        sw_now = float(s.iloc[-1])
        source_modes['SW_speed'] = 'last_valid'
        print(f'  SW fallback (last_valid): {sw_now:.0f} km/s')

print()
print('Source modes:', source_modes)

# ============================================================
# STEP 2 – Score from available components only
# ============================================================

def norm(v, lo, hi):
    if v is None: return None
    return max(0.0, min(1.0, (v - lo) / (hi - lo)))

COMPONENTS = {
    'solar_radiation_f107':    (f107_now, norm(f107_now, 60, 250)),
    'geomagnetic_activity_kp': (kp_now,   norm(kp_now,   0,   9)),
    'imf_bz_southward':        (bz_now,   norm(-(bz_now or 0) if bz_now else None, -5, 30)
                                           if bz_now is not None else None),
    'solar_wind_speed':        (sw_now,   norm(sw_now, 300, 800)),
}

available   = {k: v[1] for k, v in COMPONENTS.items() if v[1] is not None}
unavailable = [k for k, v in COMPONENTS.items() if v[1] is None]

layer0_score = round(sum(available.values()) / len(available), 4) if available else None
confidence   = round(len(available) / len(COMPONENTS), 2)

level = ('unknown'  if layer0_score is None
         else 'calm'     if layer0_score < 0.3
         else 'moderate' if layer0_score < 0.6
         else 'active')

# ============================================================
# STEP 3 – Dominant Driver
# ============================================================

def dominant_driver(kp, f107, bz, sw):
    candidates = []
    if kp   is not None and kp   >= 4:   candidates.append(('geomagnetic',    kp / 9))
    if sw   is not None and sw   >= 500:  candidates.append(('solar_wind',     sw / 800))
    if bz   is not None and bz   <= -5:   candidates.append(('imf_southward',  abs(bz) / 30))
    if f107 is not None and f107 >= 150:  candidates.append(('solar_radiation',f107 / 250))
    return max(candidates, key=lambda x: x[1])[0] if candidates else 'none'

driver = dominant_driver(kp_now, f107_now, bz_now, sw_now)

# ============================================================
# STEP 4 – Downstream expectations
# ============================================================

def downstream_expectation(driver, score, kp, bz, f107):
    exp = {
        'magnetosphere':        'calm',
        'ionosphere':           'calm',
        'schumann_resonance':   'unchanged',
        'lithosphere_currents': 'low',
    }
    if driver == 'geomagnetic' or (kp and kp >= 5):
        exp['magnetosphere']        = 'compressed / storm'
        exp['ionosphere']           = 'elevated disturbance (TEC variability)'
        exp['schumann_resonance']   = 'elevated amplitude possible'
        exp['lithosphere_currents'] = 'elevated induction currents'
    elif driver == 'imf_southward' or (bz and bz <= -5):
        exp['magnetosphere']        = 'energy coupling active'
        exp['ionosphere']           = 'elevated electron density possible'
        exp['schumann_resonance']   = 'slight modulation possible'
    elif driver == 'solar_radiation' or (f107 and f107 >= 150):
        exp['ionosphere']           = 'elevated ionization (F-layer)'
        exp['schumann_resonance']   = 'slightly elevated background activity'
    return exp

downstream = downstream_expectation(driver, layer0_score, kp_now, bz_now, f107_now)

# ============================================================
# CONSOLE OUTPUT
# ============================================================

W = 58
print('=' * W)
print('LAYER 0 – STATE ASSESSMENT')
print('=' * W)
for name, (raw_v, score_v) in COMPONENTS.items():
    if score_v is not None:
        bar = '█' * int(score_v * 20) + '░' * (20 - int(score_v * 20))
        print(f'  {name:<35}  {bar}  {score_v:.2f}')
    else:
        print(f'  {name:<35}  {"─" * 20}  n/a')
print('-' * W)
print(f'  Score ({len(available)}/{len(COMPONENTS)} components):  {layer0_score:.3f}')
print(f'  Confidence:         {confidence:.0%}')
print(f'  Level:              {level.upper()}')
print(f'  Dominant Driver:    {driver}')

# Radar chart
cats = list(available.keys())
vals = list(available.values())
if len(cats) >= 3:
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=cats + [cats[0]],
        fill='toself', fillcolor='rgba(242,166,35,0.25)',
        line=dict(color='#F2A623', width=2.5), name='Current'
    ))
    fig.add_trace(go.Scatterpolar(
        r=[0.6] * (len(cats) + 1), theta=cats + [cats[0]],
        line=dict(color='#E24B4A', dash='dot', width=1.2),
        mode='lines', name='Activity threshold'
    ))
    fig.update_layout(
        title=dict(text=f'Layer 0 – Activity Profile | Score: {layer0_score:.2f} | {level.upper()} | Confidence: {confidence:.0%}',
                   font=dict(size=14)),
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        height=430, showlegend=True,
        margin=dict(l=60, r=60, t=60, b=40)
    )
    fig.show()


Source modes: {'F10.7': 'primary', 'Kp': 'primary', 'IMF_Bz': 'primary', 'SW_speed': 'missing'}
LAYER 0 – STATE ASSESSMENT
  solar_radiation_f107                 ████████░░░░░░░░░░░░  0.41
  geomagnetic_activity_kp              ░░░░░░░░░░░░░░░░░░░░  0.00
  imf_bz_southward                     █████░░░░░░░░░░░░░░░  0.26
  solar_wind_speed                     ────────────────────  n/a
----------------------------------------------------------
  Score (3/4 components):  0.222
  Confidence:         75%
  Level:              CALM
  Dominant Driver:    none


In [9]:
# ============================================================
# EXPORT – layer0_test_state.json
# (test file)
# ============================================================

# state_summary pre-computed (no nested f-string)
_f107_str = (
    f'Solar radiation moderate (F10.7={f107_now:.0f} sfu).' if f107_now is not None and 100 <= f107_now < 150 else
    f'Solar radiation elevated (F10.7={f107_now:.0f} sfu).' if f107_now is not None and f107_now >= 150 else
    f'Solar radiation low (F10.7={f107_now:.0f} sfu).'      if f107_now is not None else
    'F10.7 unavailable.'
)
_kp_str = (
    f'Kp={kp_now:.1f} (calm).'        if kp_now is not None and kp_now < 3 else
    f'Kp={kp_now:.1f} (moderately active).' if kp_now is not None and kp_now < 5 else
    f'Kp={kp_now:.1f} (storm).'       if kp_now is not None else
    'Kp unavailable.'
)
_bz_orientation = (
    'strongly_southward' if bz_now is not None and bz_now <= -10 else
    'southward'          if bz_now is not None and bz_now <= -2  else
    'weak_southward'     if bz_now is not None and bz_now <   0  else
    'northward'          if bz_now is not None and bz_now >= 0   else
    None
)
_bz_str = (
    f'IMF Bz={bz_now:.1f} nT – ' + (
        'strongly southward, geoeffective.'      if bz_now <= -10 else
        'southward, geoeffective.'               if bz_now <= -5  else
        'weakly southward, not geoeffective.'    if bz_now <  0   else
        'northward, no coupling.'
    ) if bz_now is not None else 'IMF Bz unavailable.'
)
_sw_str  = f'Solar wind {sw_now:.0f} km/s.' if sw_now is not None else 'Solar wind speed unavailable.'
_w  = f'{kp_now:+.2f}' if kp_now  is not None else 'n/a'
_en_ref = kp_now if kp_now is not None else None

state_summary = ' '.join([
    f'Layer-0 state: {level}.',
    _f107_str, _kp_str, _bz_str, _sw_str,
    f'Data completeness: {confidence:.0%}.'
])

layer0_state = {
    'timestamp':  datetime.datetime.utcnow().isoformat() + 'Z',
    'layer': 0,
    'name':  'External Cosmic Drivers',

    'score':      layer0_score,
    'level':      level,
    'confidence': confidence,
    'score_basis': f'{len(available)}/{len(COMPONENTS)} components available',
    'missing_components': unavailable,

    'components': {
        k: round(v, 4) if v is not None else None
        for k, v in available.items()
    },

    'raw_values': {
        'F10.7_sfu':    {'value': round(f107_now, 1) if f107_now is not None else None,
                         'source': source_modes['F10.7']},
        'Kp_index':     {'value': round(kp_now,   2) if kp_now   is not None else None,
                         'source': source_modes['Kp']},
        'IMF_Bz_nT':    {'value': round(bz_now,   1) if bz_now   is not None else None,
                         'source': source_modes['IMF_Bz']},
        'SW_speed_kms': {'value': round(sw_now,   0) if sw_now   is not None else None,
                         'source': source_modes['SW_speed']},
    },

    'flags': {
        'geomagnetic_storm':     (kp_now  >= 5)   if kp_now   is not None else None,
        'solar_flux_high':       (f107_now >= 150) if f107_now is not None else None,
        'bz_strongly_southward': (bz_now  <= -10) if bz_now   is not None else None,
        'bz_geoeffective':       (bz_now  <= -5)  if bz_now   is not None else None,
        'bz_orientation':        _bz_orientation,
        'fast_solar_wind':       (sw_now  >= 500) if sw_now   is not None else None,
    },

    'dominant_driver': driver,
    'downstream_expectation': downstream,
    'state_summary': state_summary,
}

with open('../data/states/layer0_test_state.json', 'w', encoding='utf-8') as f:
    json.dump(layer0_state, f, indent=2, ensure_ascii=False)

print('../data/states/layer0_test_state.json saved')
print(json.dumps(layer0_state, indent=2, ensure_ascii=False))

../data/states/layer0_test_state.json saved
{
  "timestamp": "2026-07-23T12:11:04.785948Z",
  "layer": 0,
  "name": "External Cosmic Drivers",
  "score": 0.2224,
  "level": "calm",
  "confidence": 0.75,
  "score_basis": "3/4 components available",
  "missing_components": [
    "solar_wind_speed"
  ],
  "components": {
    "solar_radiation_f107": 0.4116,
    "geomagnetic_activity_kp": 0.0,
    "imf_bz_southward": 0.2554
  },
  "raw_values": {
    "F10.7_sfu": {
      "value": 138.2,
      "source": "primary"
    },
    "Kp_index": {
      "value": 0.0,
      "source": "primary"
    },
    "IMF_Bz_nT": {
      "value": -3.9,
      "source": "primary"
    },
    "SW_speed_kms": {
      "value": null,
      "source": "missing"
    }
  },
  "flags": {
    "geomagnetic_storm": false,
    "solar_flux_high": false,
    "bz_strongly_southward": false,
    "bz_geoeffective": false,
    "bz_orientation": "southward",
    "fast_solar_wind": null
  },
  "dominant_driver": "none",
  "downstream_expe

---
## Summary Layer 0

| Aspect | Content |
|--------|--------|
| **Data Sources** | NOAA SWPC (Kp, Solar Wind, IMF, X-Ray, SSN, F10.7) |
| **Update Rate** | 1-minute (Kp, Wind, Mag), monthly means (F10.7, SSN) |
| **Output** | `layer0_test_state.json` – score, raw values, flags |
| **→ Layer 4** | UV/X-Ray → ionosphere ionization rate |
| **→ Layer 5** | Solar wind / IMF Bz → Global Electric Circuit |
| **→ Layer 1** | Geomagnetic induction → lithospheric currents |

> **Next step:** `layer1_planetary_body.ipynb`